# Task 1

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            background-color: #f9f9f9;
            color: #333;
            padding: 20px;
        }
        h1, h2, h3 {
            color: #2c3e50;
        }
        .important {
            background-color: #fffae6;
            border-left: 6px solid #f1c40f;
            padding: 10px;
            margin-bottom: 20px;
        }
        .submission {
            background-color: #e8f4fd;
            border-left: 6px solid #2196f3;
            padding: 10px;
            margin-bottom: 20px;
        }
        ul {
            margin-left: 20px;
        }
        .tips {
            background-color: #e8f5e9;
            border-left: 6px solid #4caf50;
            padding: 10px;
            margin-top: 20px;
        }
    </style>
</head>
<body>

<h1>NER Fine-Tuning Task</h1>

<div class="important">
    <h2>Important Setup:</h2>
    <ul>
        <li>Make sure to use <strong>GPU acceleration</strong> if available in your environment for faster training</li>
        <li>You can use <strong>Kaggle</strong> which offers free GPU access for this task</li>
        <li>Save your work frequently and ensure all files are saved properly</li>
    </ul>
</div>

<div class="submission">
    <h2>Submission Requirements:</h2>
    <ul>
        <li>Send your completed work as files (Jupyter Notebook, Python files, and any additional files)</li>
        <li><strong>Include:</strong> Your notebook, any custom datasets, model files, and documentation</li>
        <li>Ensure all code is runnable and well-documented</li>
    </ul>
</div>

<h2>Your Tasks:</h2>
<p><strong>Fine-tune a pretrained model on a Named Entity Recognition (NER) task.</strong></p>

<h3>Keep in mind:</h3>
<ul>
    <li><strong>Loss function</strong></li>
    <li><strong>Optimizer</strong></li>
    <li><strong>Weight initialization</strong> (if applicable)</li>
    <li><strong>Splitting data</strong> into train, validation, and test sets</li>
    <li><strong>Evaluation metrics:</strong> e.g., F1-score, precision, recall</li>
</ul>

<h3>Allowed Resources:</h3>
<p>You are allowed to use:</p>
<ul>
    <li>AI tools</li>
    <li>YouTube tutorials</li>
    <li>Online documentation, blogs, and forums</li>
</ul>
<p><em>In short: Feel free to use the internet for research and implementation.</em></p>

<h3>Dataset Requirements:</h3>
<ul>
    <li><strong>You need to collect or use an Azerbaijani dataset.</strong></li>
</ul>

<h3>Before Starting, Research These Topics:</h3>
<ul>
    <li><strong>Tokenization</strong></li>
    <li><strong>NER labeling schemes</strong> (such as <strong>BIO encoding</strong>)</li>
    <li><strong>Pretrained transformer models for NER:</strong> e.g., BERT, RoBERTa, etc.</li>
</ul>

<h3>Libraries and Frameworks:</h3>
<ul>
    <li>You are allowed to use existing libraries and frameworks:</li>
    <li>Examples: <strong>Hugging Face Transformers, spaCy</strong>, etc.</li>
</ul>

<div class="tips">
    <h3>Recommended:</h3>
    <ul>
        <li>Document your experiment steps, including <strong>hyperparameter choices</strong>, <strong>training logs</strong>, and <strong>evaluation results</strong>.</li>
        <li>If possible, <strong>visualize your results</strong> (e.g., using matplotlib or seaborn for learning curves, confusion matrices, etc.).</li>
    </ul>
</div>

</body>
</html>

In [ ]:
# For example, here is one library that can be used to complete this task (finding a different one, or building fine-tuning from scratch in pytorch is also an option)
# https://flairnlp.github.io/docs/category/tutorial-2-training-models

The code was run in Google Colab environment.

## Code

### Install required dependencies


After running 2 cells below, you may need to restart the session, if the code is being run in Colab.

In [ ]:
# for Jupyter:
!pip install -U datasets
!pip install transformers tokenizers seqeval -q

In [ ]:
!pip install evaluate

### Required imports

In [ ]:
import datasets
import numpy as np
from transformers import DataCollatorForTokenClassification
from transformers import BertTokenizerFast
from transformers import AutoModelForTokenClassification

### Loading data

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset

ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")

In [ ]:
ds.shape

Our data does not have val/test split in the beginning. We need to split it before we apply transformations as needed.

### train/val/test split

In [ ]:
from datasets import DatasetDict

# First split train into train + temp (80% train, 20% temp)
train_testvalid = ds['train'].train_test_split(test_size=0.2, seed=42)

# Then split temp into validation and test (50% each of the 20%)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Combine into final DatasetDict
ds = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],  # Note: 'train' here refers to the first split of test_valid
    'test': test_valid['test']
})

print(ds)

In [ ]:
ds['train']

In [ ]:
print(ds['train'].features)
# Should show something like: {'ner_tags': Value('string')}

All features are datatype of Value('string').
\
NER tags and tokens need to be changed to be processed as Sequences of Lists.
\
index is not needed in this case, as they do not hold any useful information

In [ ]:
ds['train']['ner_tags']

NER tags are stored as strings, in following parts, it's been taken care of.

### preprocessing data

In [ ]:
ds = ds.select_columns(['tokens', 'ner_tags'])

ds

In following cells of code, I tried to make dictionaries in forms of {NER tag: Description} and {index: NER tag}, and vice-versa.

In [ ]:
# Own description in HuggingFace

chars = '''
0: O: Outside any named entity
1: PERSON: Names of individuals
2: LOCATION: Geographical locations, both man-made and natural
3: ORGANISATION: Names of companies, institutions
4: DATE: Dates or periods
5: TIME: Times of the day
6: MONEY: Monetary values
7: PERCENTAGE: Percentage values
8: FACILITY: Buildings, airports, etc.
9: PRODUCT: Products and goods
10: EVENT: Events and occurrences
11: ART: Artworks, titles of books, songs
12: LAW: Legal documents
13: LANGUAGE: Languages
14: GPE: Countries, cities, states
15: NORP: Nationalities or religious or political groups
16: ORDINAL: Ordinal numbers
17: CARDINAL: Cardinal numbers
18: DISEASE: Diseases and medical conditions
19: CONTACT: Contact information, e.g., phone numbers, emails
20: ADAGE: Proverbs, sayings
21: QUANTITY: Measurements and quantities
22: MISCELLANEOUS: Miscellaneous entities
23: POSITION: Professional or social positions
24: PROJECT: Names of projects or programs
'''

In [ ]:
arbitrary_list = []
for i, char in enumerate(chars.split('\n')):
  arbitrary_list.append(char.split(':'))

arbitrary_list[1:-1]

In [ ]:
ner_tags_id_to_char = {}
ner_tags_char_description = {}
for lst in arbitrary_list[1:-1]:
  ner_tags_id_to_char[lst[0]] = lst[1].strip()
  ner_tags_char_description[lst[1].strip()] = lst[2].strip()

In [ ]:
ner_tags_char_description, ner_tags_id_to_char

In [ ]:
type(ds['train']['ner_tags'][3])

In [ ]:
# Some rows in dataset were type None.
# This datapoints will be a headache, so let's look at how many are there, and get rid of them.

none_indices = [i for i, x in enumerate(ds['train']['ner_tags']) if x is None]
print(f"Found {len(none_indices)} rows with 'ner_tags' = None")

In [ ]:
# Filter out rows where 'ner_tags' is None
ds['train'] = ds['train'].filter(lambda example: example['ner_tags'] is not None)

In [ ]:
none_indices = [i for i, x in enumerate(ds['train']['ner_tags']) if x is None]
print(f"Found {len(none_indices)} rows with 'ner_tags' = None")

Another important part is to clean the data.
\
In Named Entity Recognition, models expect to work with batches, like sequences of lists.
\
\
We have tokens (words) and NER tags. NER tags must be **integers**, because they represent the **id's**, and **tokens** should be properly split into words.
\
\
Currently, they look like the following:

In [ ]:
ds['train']['tokens']

In [ ]:
ds['train']['ner_tags']

In [ ]:
unique_elements = []
for i in arbitrary_list[1:-1]:
  unique_elements.append(i[1].strip())

unique_elements

In [ ]:
import ast
import numpy as np
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# proper conversion of 'token' features
def convert_string_to_list(example):
    example['tokens'] = ast.literal_eval(example['tokens'])
    return example

# proper conversion of 'ner_tags' features
# 1. First convert string lists to actual lists of integers
def convert_tags(example):
    tag_list = ast.literal_eval(example['ner_tags'])  # Convert string to list
    return {'ner_tags': [int(tag) for tag in tag_list]}  # Ensure all elements are integers

# 2. Define your NER tag classes (customize these to match your labels)
tag_names = unique_elements

# 3. Apply conversion and casting
ds = ds.map(convert_string_to_list)
ds = ds.map(convert_tags, batched=False)

ds = ds.cast_column('ner_tags', Sequence(feature=ClassLabel(names=tag_names)))
ds = ds.cast_column('tokens', Sequence(feature=Value(dtype='string')))

In [ ]:
# Below is the Correct datatype.

ds['train'].features

In [ ]:
# Now, they look like this:

ds['train']['tokens']

In [ ]:
ds['train']['ner_tags']

Conversion was succesful!

### Count of NER tags

In [ ]:
ner_tag_count = {}

for example in ds['train']:
  for ner_tag in example['ner_tags']:
    ner_tag_count[ner_tag] = ner_tag_count.get(ner_tag, 0) + 1

In [ ]:
ner_tag_count

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Keys will be more understandable, will be used in plotting

ner_tag_count_id_to_char = {ner_tags_id_to_char[str(i)]: j for i, j in zip(ner_tag_count, ner_tag_count.values())}
ner_tag_count_id_to_char

In [ ]:
# Changing the key name from 'O' to 'Undefined'.

dict_copy = ner_tag_count_id_to_char.copy()
dict_copy['Undefined'] = dict_copy.pop('O')
dict_copy

In [ ]:
import pandas as pd

In [ ]:
# Plotting

d = {
    'Named Entity': dict_copy.keys(),
    'Count': dict_copy.values()
}

df = pd.DataFrame(d)
df.sort_values(by='Count', ascending=False).plot.bar(x='Named Entity', y='Count', figsize=(14, 6), rot=45)

plt.show()

Too much 'Undefined', which is id=0, meaning that we have "Outside any named entity" tags quite a lot.

In [ ]:
# tokenizer is bert-base-uncased, which will be fine-tuned.

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
ds['train'][0]

In [ ]:
# id 101 and 102, are start of sequence and end of sequence tokens(Also seen as [CLS] and [SEP]).

example_text = ds['train'][0]
tokenized_input = tokenizer(example_text['tokens'], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])

word_ids = tokenized_input.word_ids()

print(word_ids)

In [ ]:
tokenized_input

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])
tokens

In [ ]:
len(tokens)

In [ ]:
ds['train'][0]['ner_tags']

Count of *ner_tags* and *tokenized inputs* are different.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )

    labels = []
    for i, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens get -100
            if word_idx is None:
                label_ids.append(-100)
            # New word that exists in original sequence
            elif word_idx < len(label_seq):
                label_ids.append(label_seq[word_idx])
            # Word index out of bounds
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        # Ensure labels match tokenized length
        if len(label_ids) != len(tokenized_inputs["input_ids"][i]):
            label_ids = label_ids[:len(tokenized_inputs["input_ids"][i])] + [-100] * (len(tokenized_inputs["input_ids"][i]) - len(label_ids))

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
ds['train'][4:5]

In [ ]:
 q = tokenize_and_align_labels(ds['train'][4:5])
 print(q)

In [ ]:
for token, label in zip(tokenizer.convert_ids_to_tokens(q['input_ids'][0]), q['labels'][0]):
  print(f"{token: <40} {label}")

In [ ]:
ds

In [ ]:
tokenized_datasets = ds.map(tokenize_and_align_labels, batched=True)

In [ ]:
tokenized_datasets['train'][0]

In [ ]:
len(unique_elements)

In [ ]:
# Defining model

model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=25)

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    "test-ner",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
from evaluate import load

In [ ]:
metric = load("seqeval")

In [ ]:
example = ds['train'][0]

In [ ]:
label_list = ds['train'].features['ner_tags'].feature.names

In [ ]:
label_list

In [ ]:
for i in example['ner_tags']:
  print(i)

In [ ]:
labels = [label_list[i] for i in example['ner_tags']]
labels

In [ ]:
# trial

metric.compute(predictions=[labels], references=[labels])

In [ ]:
def compute_metrics(eval_preds):
  pred_logits, labels = eval_preds

  pred_logits = np.argmax(pred_logits, axis=2)


  predictions = [
      [label_list[eval_preds] for (eval_preds, l) in zip(prediction, label) if l != -100]
      for (prediction, label) in zip(pred_logits, labels)
  ]

  true_labels = [
      [label_list[l] for (eval_preds, l) in zip(prediction, label) if l != -100]
      for (prediction, label) in zip(pred_logits, labels)
  ]

  results = metric.compute(predictions=predictions, references=true_labels)

  return {
      "precision": results["overall_precision"],
      "recall": results["overall_recall"],
      "f1": results["overall_f1"],
      "accuracy": results["overall_accuracy"]
  }

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("ner_model")

In [ ]:
tokenizer.save_pretrained("tokenizer")

In [ ]:
id2label = {
    str(i): label for i, label in enumerate(label_list)
}

label2id = {
    label: str(i) for i, label in enumerate(label_list)
}

In [ ]:
id2label

In [ ]:
label2id

In [ ]:
import json

In [ ]:
config = json.load(open("ner_model/config.json"))
config["id2label"] = id2label
config["label2id"] = label2id
json.dump(config, open("ner_model/config.json", "w"))

In [ ]:
model_fine_tuned = AutoModelForTokenClassification.from_pretrained("ner_model")

In [ ]:
from transformers import pipeline

In [ ]:
nlp = pipeline("ner", model=model_fine_tuned, tokenizer=tokenizer)

example = "Yoxlama məqsədli bir yazı. Model bu yazdığımı tokenlərə necə ayıracaq?"

ner_results = nlp(example)

print(ner_results)

# Task 2

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            background-color: #f4f8fb;
            color: #2c3e50;
            padding: 20px;
        }
        h1, h2, h3 {
            color: #34495e;
        }
        .important {
            background-color: #fff3cd;
            border-left: 6px solid #f0ad4e;
            padding: 10px;
            margin-bottom: 20px;
        }
        .bonus {
            background-color: #e0f7fa;
            border-left: 6px solid #00bcd4;
            padding: 10px;
            margin-top: 20px;
        }
        ul {
            margin-left: 20px;
        }
    </style>
</head>
<body>

<h1>Task 2: Voice Activity Detection (VAD) & Speech Emotion Recognition (SER)</h1>

<h2>Your Goal:</h2>
<p>Create a <strong>working example</strong> that demonstrates:</p>
<ul>
    <li><strong>Voice Activity Detection (VAD):</strong> Detecting speech vs. silence in an audio file.</li>
    <li><strong>Speech Emotion Recognition (SER):</strong> Identifying emotional tone in speech (e.g., happy, sad, angry, neutral, etc.).</li>
</ul>

<h3>Audio Input:</h3>
<p>You can use <strong>any audio file</strong> (your voice, downloaded samples, or synthetic speech). The focus is on making a <strong>working example</strong>.</p>

<div class="important">
    <h3>Bonus:</h3>
    <p>If your example works with <strong>Azerbaijani language audio</strong>, you will receive <strong>extra points</strong></p>
</div>

<h3>RRecommended (but not required) Libraries & Tools:</h3>
<ul>
    <li><strong>For VAD:</strong>
        <ul>
            <li>Silero VAD</li>
            <li>WebRTC VAD</li>
            <li>py-webrtcvad</li>
        </ul>
    </li>
    <li><strong>For SER:</strong>
        <ul>
            <li>SpeechBrain</li>
            <li>OpenSMILE (via Python)</li>
            <li>Pretrained Hugging Face SER models</li>
        </ul>
    </li>
</ul>

<h3>Allowed Resources:</h3>
<ul>
    <li>You are <strong>allowed and encouraged</strong> to use any open-source tools, prebuilt models, AI tools, YouTube tutorials, and online guides.</li>
</ul>
</body>
</html>


In [ ]:
# Note:
# The main idea of this task is to understand your interests and see what you can accomplish independently,
# including using AI if you wish. In real projects,
# some research and effort are usually needed to develop working examples,
# so we encourage you to explore and do your best—there’s no pressure to be perfect!

# You can submit your work even if it is not fully completed.

## Voice Activity Detection

In [ ]:
!pip install funasr

In [ ]:
paths[:5]

In [ ]:
paths = []
labels = []
for dirname, _, filenames in os.walk('/root/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/versions/1'):
  for filename in filenames:
      paths.append(os.path.join(dirname, filename))
      label = filename.split('_')[-1]
      label = label.split('.')[0]
      labels.append(label.lower())
  if len(paths) == 2800:
    break
print('Dataset is Loaded')

In [ ]:
# from funasr import AutoModel
# import torchaudio

# # Initialize the VAD model
# model = AutoModel(model="fsmn-vad", model_revision="v2.0.4")

# wav_file = "/root/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/versions/1/TESS Toronto emotional speech set data/YAF_sad/YAF_gap_sad.wav"

# from funasr import AutoModel

# Initialize with vad_inference_pipeline
# model = AutoModel(model="fsmn-vad",
#                  model_revision="v2.0.4",
#                  vad_inference_pipeline=True)  # ← Critical for VAD output

# wav_file = "/your/audio/file.wav"  # Update path

# This will now return speech segments
model.generate(input=paths[:3])


# Typical output structure:
# [
#   {
#     "text": "(speech segments in text)",
#     "timestamp": [[start1, end1], [start2, end2],...],
#     "raw_text": "detailed output"
#   }
# ]


## Speech Recognition

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess")

print("Path to dataset files:", path)

In [ ]:
paths = []
labels = []
for dirname, _, filenames in os.walk('/root/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/versions/1'):
  for filename in filenames:
      paths.append(os.path.join(dirname, filename))
      label = filename.split('_')[-1]
      label = label.split('.')[0]
      labels.append(label.lower())
  if len(paths) == 2800:
    break
print('Dataset is Loaded')

In [ ]:
len(paths)

In [ ]:
paths[:5]


In [ ]:
df = pd.DataFrame()
df['speech'] = paths
df['label'] = labels
df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
sns.countplot(df, x='label')

In [ ]:
def waveplot(data, sr, emotion):
    plt.figure(figsize=(10,4))
    plt.title(emotion, size=20)
    librosa.display.waveshow(data, sr=sr)
    plt.show()

def spectogram(data, sr, emotion):
    x = librosa.stft(data)
    xdb = librosa.amplitude_to_db(abs(x))
    plt.figure(figsize=(11,4))
    plt.title(emotion, size=20)
    librosa.display.specshow(xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar()

In [ ]:
emotion = 'fear'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)


In [ ]:
emotion = 'angry'
path = np.array(df['speech'][df['label']==emotion])[1]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'disgust'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'neutral'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'sad'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'ps'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'happy'
path = np.array(df['speech'][df['label']==emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
def extract_mfcc(filename):
    y, sr = librosa.load(filename, duration=3, offset=0.5)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    return mfcc

In [ ]:
extract_mfcc(df['speech'][0])

In [ ]:
X_mfcc = df['speech'].apply(lambda x: extract_mfcc(x))

In [ ]:
X_mfcc

In [ ]:
X = [x for x in X_mfcc]
X = np.array(X)
X.shape

In [ ]:
## input split
X = np.expand_dims(X, -1)
X.shape

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
y = enc.fit_transform(df[['label']])

In [ ]:
y = y.toarray()

In [ ]:
y.shape

In [ ]:
### LSTM Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np

# Define the LSTM model
class LSTMModel(nn.Module):
  def __init__(self, input_size=1, hidden_size=256, num_layers=1, num_classes=7):
    super(LSTMModel, self).__init__()
    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True
    )
    self.dropout1 = nn.Dropout(0.2)
    self.fc1 = nn.Linear(hidden_size, 128)
    self.dropout2 = nn.Dropout(0.2)
    self.fc2 = nn.Linear(128, 64)
    self.dropout3 = nn.Dropout(0.2)
    self.fc3 = nn.Linear(64, num_classes)
    self.relu = nn.ReLU()
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    # LSTM layer
    lstm_out, _ = self.lstm(x)
    # Only take the output from the final time step
    lstm_out = lstm_out[:, -1, :]

    # Fully connected layers with dropout
    x = self.dropout1(lstm_out)
    x = self.relu(self.fc1(x))
    x = self.dropout2(x)
    x = self.relu(self.fc2(x))
    x = self.dropout3(x)
    x = self.fc3(x)
    return self.softmax(x)

# Convert numpy arrays to PyTorch tensors
X_tensor = torch.FloatTensor(X)  # Shape: (batch_size, seq_length=40, input_size=1)
y_tensor = torch.LongTensor(np.argmax(y, axis=1))  # Convert one-hot to class indices

# Create dataset and split for validation
dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Initialize model
model = LSTMModel(input_size=1, hidden_size=256, num_classes=7)
print(model)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

import matplotlib.pyplot as plt

# Initialize lists to store metrics
train_losses = []
train_accs = []
val_losses = []
val_accs = []

num_epochs = 15

# Modified training loop to store metrics
for epoch in range(num_epochs):
  # Training
  model.train()
  train_loss = 0.0
  train_correct = 0
  for inputs, labels in train_loader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    train_correct += (predicted == labels).sum().item()

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
      for inputs, labels in val_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        val_correct += (predicted == labels).sum().item()

    # Calculate metrics
  train_epoch_loss = train_loss/len(train_loader)
  train_epoch_acc = train_correct/train_size
  val_epoch_loss = val_loss/len(val_loader)
  val_epoch_acc = val_correct/val_size

    # Store metrics
  train_losses.append(train_epoch_loss)
  train_accs.append(train_epoch_acc)
  val_losses.append(val_epoch_loss)
  val_accs.append(val_epoch_acc)

  print(f'Epoch {epoch+1}/{num_epochs}:')
  print(f'Train Loss: {train_epoch_loss:.4f}, Acc: {train_epoch_acc:.4f}')
  print(f'Val Loss: {val_epoch_loss:.4f}, Acc: {val_epoch_acc:.4f}\n')

In [ ]:
# Plotting
plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get all predictions
all_preds = []
all_labels = []
with torch.no_grad():
  for inputs, labels in val_loader:
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.numpy())
    all_labels.extend(labels.numpy())

# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))